# Diagnostics of the pygenic initial conditions

This notebook combines `plot_matter_pk.py`, `plot_velocity.py` and `plot_mass.py`.

1. **Matter power spectrum**: measured $P(k)$ of each IC and of a reference, against the linear input.
2. **Velocities and momenta**: distribution of the particle speed $|v|$, or of the momentum per mean
   particle mass $(m/\langle m\rangle)\,|v|$, against the reference.
3. **Particle masses**: histogram of the masses of an IC with `PerturbMass = 1`.

Each IC is an MP-GenIC / pygenic BigFile (e.g. `ICS_grid/IC`; the output directory `ICS_grid` works too).
Run the notebook from `runs/test_pygenic`. Set which ICs to compare in the next cell.

In [ ]:
# ICs to compare against the reference (at most 4 in one figure)
ICS = ["ICS_grid", "ICS_glass", "ICS_test"]
REF = "ICS_ref_grid/IC"
# IC with perturbed masses (PerturbMass = 1), for the mass histogram
MASS_IC = "ICS_test"
# Save each figure as a PNG next to the notebook as well as showing it
SAVE = True

## Shared helpers

In [ ]:
import glob
import os

import bigfile
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

CM_PER_MPC = 3.085678e24
GAS, DM = 0, 1

# Colours: the reference is neutral ink, each compared IC takes the next hue in fixed order.
SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_SECONDARY = "#52514e"
GRID = "#e4e3df"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update({"font.size": 11, "axes.edgecolor": INK_SECONDARY,
                     "axes.labelcolor": INK, "text.color": INK,
                     "xtick.color": INK_SECONDARY, "ytick.color": INK_SECONDARY})


def open_ic(path):
    """Open an IC BigFile. Accepts the snapshot itself or the directory holding IC/."""
    path = path.rstrip("/")
    if not os.path.isdir(os.path.join(path, "Header")):
        inner = os.path.join(path, "IC")
        if not os.path.isdir(os.path.join(inner, "Header")):
            raise FileNotFoundError(f"{path}: no BigFile snapshot here (no Header block)")
        path = inner
    return path, bigfile.File(path)


def read_header(f):
    attrs = f["Header"].attrs
    npart = np.asarray(attrs["TotNumPart"])
    return {
        "box": float(attrs["BoxSize"][0]),
        "redshift": float(attrs["Redshift"][0]),
        "mpc": float(attrs["UnitLength_in_cm"][0]) / CM_PER_MPC,  # length unit in Mpc/h
        "masstable": np.asarray(attrs["MassTable"]),
        "types": [pt for pt in (GAS, DM) if npart[pt] > 0],
        # pygenic says in the header whether the Mass blocks belong to this IC.
        "perturbmass": int(attrs["PerturbMass"][0]) if "PerturbMass" in attrs else None,
        "ngrid": round(max(npart[GAS], npart[DM]) ** (1 / 3)),
    }


def label_of(icpath):
    """Name an IC by its output directory, e.g. ICS_ref/IC -> ICS_ref."""
    return os.path.basename(os.path.dirname(os.path.abspath(icpath))) or icpath


def style_axes(ax):
    ax.set_facecolor(SURFACE)
    ax.grid(color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)


def finish(fig, out):
    if SAVE and out:
        fig.savefig(out, dpi=150, facecolor=SURFACE)
        print(f"saved {out}")
    plt.show()

## 1. Matter power spectrum

There is one column of panels for the total matter and, if the ICs have gas, one for the gas alone.
The top panel shows the measured $P(k)$ of each IC and of the reference. The bottom panel shows each of
them divided by the linear input, averaged over the same modes as the measurement.

The linear input for the total matter is the `inputspec_*.txt` the generator writes next to the IC.
For the gas it is that spectrum times the square of the baryon transfer function (`DELTA_BAR` in the
`ICTransfers` block of the IC, which is relative to the total matter).

The total matter density is the mass-weighted sum of gas (type 0) and dark matter (type 1). Particles
are weighted by their `Mass` block if they have one (`PerturbMass = 1`) and by the header `MassTable`
otherwise. A `Mass` block is ignored, with a warning, if the header says `PerturbMass = 0`.

The density is assigned with a TSC kernel on a mesh four times finer than the particle grid, and the
kernel is deconvolved. Cloud-in-cell on a coarser mesh is biased at the percent level for particles on
a grid (see `genic/pygenic/README.md`).

In [ ]:
PK_TITLES = {"total": "Total matter", "gas": "Gas"}


def tsc_mass_mesh(pos, mass, box, nmesh):
    """Mass on a periodic nmesh^3 mesh, assigned with the triangular shaped cloud kernel."""
    x = (pos % box) / box * nmesh
    centre = np.rint(x).astype(np.int64)
    d = x - centre
    # Weights of the cells at centre - 1, centre and centre + 1, for each axis.
    w = np.stack([0.5 * (0.5 - d) ** 2, 0.75 - d ** 2, 0.5 * (0.5 + d) ** 2])
    rho = np.zeros(nmesh ** 3)
    for ix in range(3):
        for iy in range(3):
            for iz in range(3):
                idx = (centre + [ix - 1, iy - 1, iz - 1]) % nmesh
                flat = (idx[:, 0] * nmesh + idx[:, 1]) * nmesh + idx[:, 2]
                weight = mass * w[ix, :, 0] * w[iy, :, 1] * w[iz, :, 2]
                rho += np.bincount(flat, weights=weight, minlength=nmesh ** 3)
    return rho.reshape((nmesh,) * 3)


def power_spectrum(delta, box, kmax_fund, theory=None):
    """Spherically averaged P(k) of an overdensity mesh, with the TSC window deconvolved.

    Bins are one fundamental mode wide, up to kmax_fund (in units of the fundamental).
    Returns k (in 1 / units of box) and P (in units of box^3).

    theory is an optional function P(k). It is averaged over the same modes as the
    measurement and returned as a third value (None without theory). P(k) is steep, so its
    average over a bin is not P at the mean k: in the lowest bin they differ by about 10%.
    """
    n = delta.shape[0]
    dk = np.fft.rfftn(delta) / n ** 3
    kf = np.fft.fftfreq(n, d=1.0 / n)
    kz = np.fft.rfftfreq(n, d=1.0 / n)
    kx, ky, kzz = kf[:, None, None], kf[None, :, None], kz[None, None, :]
    window = (np.sinc(kx / n) * np.sinc(ky / n) * np.sinc(kzz / n)) ** 3
    p = np.abs(dk / window) ** 2 * box ** 3
    kmag = np.sqrt(kx ** 2 + ky ** 2 + kzz ** 2)
    # The real FFT stores half of the modes: those with 0 < kz < n/2 stand for two.
    weight = np.broadcast_to(np.where((kzz > 0) & (kzz < n / 2), 2.0, 1.0), kmag.shape)
    edges = np.arange(0.5, kmax_fund + 1, 1.0)
    nbins = len(edges) - 1
    ibin = np.digitize(kmag, edges)
    used = (ibin >= 1) & (ibin <= nbins)
    ibin, weight, kmag, p = ibin[used] - 1, weight[used], kmag[used] * 2 * np.pi / box, p[used]

    def bin_sum(values):
        return np.bincount(ibin, weights=values * weight, minlength=nbins)

    count = bin_sum(1.0)
    good = count > 0
    ptheory = None if theory is None else bin_sum(theory(kmag))[good] / count[good]
    return bin_sum(kmag)[good] / count[good], bin_sum(p)[good] / count[good], ptheory


def read_input_spectrum(icpath, head):
    """The linear matter power the generator was given, from inputspec_*.txt next to the IC.
    Returns k [h/Mpc] and P [(Mpc/h)^3], or None if the file is not there."""
    found = sorted(glob.glob(os.path.join(os.path.dirname(icpath), "inputspec_*.txt")))
    if not found:
        return None
    k, p = np.loadtxt(found[0], unpack=True)  # k is in h/Mpc, P in internal length^3
    return k, p * head["mpc"] ** 3


def linear_theory(f, spec):
    """The linear input P(k) of each species, as functions of k [h/Mpc] giving (Mpc/h)^3.

    spec = (k, P) is the total matter input. The gas has the baryon transfer function of
    the IC applied. Without a transfer table the gas follows the total matter."""
    logk_in, logp_in = np.log(spec[0]), np.log(spec[1])

    def total(k):  # log-log interpolation of the table
        return np.exp(np.interp(np.log(k), logk_in, logp_in))

    # f.blocks does not list the columns of ICTransfers, so look on disk.
    if not os.path.isdir(os.path.join(f.basename, "ICTransfers", "DELTA_BAR")):
        return {"total": total, "gas": total}
    tlogk = f["ICTransfers/logk"][:]  # log10 of k in h/Mpc
    tbar = f["ICTransfers/DELTA_BAR"][:]  # delta_b / delta_tot
    if len(tlogk) == 0:
        return {"total": total, "gas": total}

    def gas(k):  # the transfer function is held constant beyond its table, as in the generator
        return total(k) * np.interp(np.log10(k), tlogk, tbar) ** 2

    return {"total": total, "gas": gas}


def measure(f, head, nmesh, kmax_fund, theory, species):
    """P(k) of each species of an IC, and its linear input in the same bins.

    Returns {species: (k [h/Mpc], P [(Mpc/h)^3], binned linear input [(Mpc/h)^3])}, and
    whether the particles carry their own masses."""
    box, mpc = head["box"], head["mpc"]
    mesh = {}
    own_mass = False
    for pt in head["types"]:
        pos = f[f"{pt}/Position"][:]
        has_mass = f"{pt}/Mass" in f.blocks
        if has_mass and head["perturbmass"] == 0:
            # Left behind by an earlier PerturbMass = 1 run into the same directory:
            # writing an IC does not remove blocks that the new run does not make.
            print(f"warning: {f.basename}: ignoring the {pt}/Mass block, "
                  f"because the header says PerturbMass = 0")
            has_mass = False
        if has_mass:
            mass = f[f"{pt}/Mass"][:].astype("f8")
            own_mass = True
        else:
            mass = np.full(len(pos), head["masstable"][pt])
        mesh[pt] = tsc_mass_mesh(pos, mass, box, nmesh)
    rho = {"total": sum(mesh.values()), "gas": mesh.get(GAS)}
    out = {}
    for name in species:
        k, p, pin = power_spectrum(rho[name] / rho[name].mean() - 1, box, kmax_fund,
                                   lambda kk, name=name: theory[name](kk / mpc))
        out[name] = (k / mpc, p * mpc ** 3, pin)
    return out, own_mass


def plot_matter_pk(ics, ref="ICS_ref_grid/IC", nmesh=None, ylim=(0.8, 1.2),
                   out="matter_pk_comparison.png"):
    """Compare the power spectra of the ICs and the reference with linear theory.

    nmesh: mesh cells per side (default: 4 x the particle grid).
    ylim: range of the ratio panels. The noise of a glass exceeds any fixed range near the
    Nyquist frequency."""
    if len(ics) > len(SERIES):
        raise ValueError(f"at most {len(SERIES)} ICs can be compared in one figure")

    refpath, reffile = open_ic(ref)
    refhead = read_header(reffile)
    opened = [open_ic(p) for p in ics]
    heads = [read_header(f) for _, f in opened]
    for (path, _), head in zip(opened, heads):
        for key in ("box", "redshift", "mpc"):
            if not np.isclose(head[key], refhead[key]):
                raise ValueError(f"{path} and {refpath} differ in {key}: "
                                 f"{head[key]:g} vs {refhead[key]:g}")

    # The gas gets its own column only if every IC has gas.
    species = ["total"]
    if all(GAS in h["types"] for h in heads + [refhead]):
        species.append("gas")

    # Measure everything on one mesh and stop at the Nyquist frequency of the coarsest
    # particle grid, so that all spectra share the same k bins.
    ngrid_min = min(h["ngrid"] for h in heads + [refhead])
    nmesh = nmesh or 4 * max(h["ngrid"] for h in heads + [refhead])
    kmax_fund = min(ngrid_min, nmesh) // 2
    box_mpc = refhead["box"] * refhead["mpc"]
    knyq = np.pi * ngrid_min / box_mpc

    # Each IC is compared with its own linear input, or the reference's if it has none.
    reflabel = label_of(refpath)
    refspec = read_input_spectrum(refpath, refhead)
    specs = [read_input_spectrum(path, head) or refspec for (path, _), head in zip(opened, heads)]
    if refspec is None:
        refspec = next((s for s in specs if s is not None), None)
        specs = [s or refspec for s in specs]
    if refspec is None:
        raise FileNotFoundError("no inputspec_*.txt next to any of the ICs: no linear input to compare with")

    # (label, {species: (k, P, binned linear input)}, has its own masses); the reference is last.
    measured = []
    for (path, f), head, spec in zip(opened, heads, specs):
        measured.append((label_of(path),)
                        + measure(f, head, nmesh, kmax_fund, linear_theory(f, spec), species))
    reftheory = linear_theory(reffile, refspec)
    measured.append((f"{reflabel} (reference)",)
                    + measure(reffile, refhead, nmesh, kmax_fund, reftheory, species))
    kref = measured[-1][1]["total"][0]

    half = kref < knyq / 2
    print(f"box {box_mpc:g} Mpc/h, z = {refhead['redshift']:g}, TSC mesh {nmesh}^3, "
          f"particle Nyquist {knyq:.2f} h/Mpc")
    for name in species:
        print(f"{PK_TITLES[name]}: measured P(k) / linear input")
        for label, spectra, own_mass in measured:
            _, p, pin = spectra[name]
            ratio = p / pin
            print(f"  {label}{' (masses from the Mass block)' if own_mass else ''}: "
                  f"k < k_Nyq/2: mean {ratio[half].mean():.4f}, "
                  f"max |ratio - 1| {np.abs(ratio[half] - 1).max():.4f}; "
                  f"all k: mean {ratio.mean():.4f}, max |ratio - 1| {np.abs(ratio - 1).max():.4f}")

    ncols = len(species)
    fig, axes = plt.subplots(2, ncols, figsize=(1.5 + 6.5 * ncols, 7.5), sharex=True,
                             sharey="row", squeeze=False, facecolor=SURFACE,
                             gridspec_kw={"height_ratios": [2, 1]}, constrained_layout=True)

    # The reference is drawn last, dashed and in neutral ink, on top of the compared ICs.
    styles = [("-", color, 2) for color in SERIES[:len(measured) - 1]] + [("--", INK, 1.5)]
    kin = refspec[0]
    kin = kin[(kin > kref.min() / 1.3) & (kin < knyq * 1.05)]
    for name, (ax1, ax2) in zip(species, axes.T):
        ax1.loglog(kin, reftheory[name](kin), ":", color=INK_SECONDARY, lw=1.5, label="linear input")
        for (label, spectra, _), (ls, color, lw) in zip(measured, styles):
            k, p, pbinned = spectra[name]
            ax1.loglog(k, p, ls, color=color, lw=lw, label=label)
            ax2.semilogx(k, p / pbinned, ls, color=color, lw=lw)
        band = ax2.axhspan(0.99, 1.01, color=GRID, lw=0, zorder=0)
        ax2.axhline(1, color=INK_SECONDARY, lw=1, ls=":")
        ax1.set_title(PK_TITLES[name], fontsize=11, loc="left")
        ax2.set_xlabel(r"$k$  [$h$/Mpc]")
        for ax in (ax1, ax2):
            style_axes(ax)

    # Every panel uses the same line styles, so one legend serves them all.
    axes[0, 0].set_ylabel(r"$P(k)$  [(Mpc/$h$)$^3$]")
    axes[0, 0].legend(frameon=False)
    axes[1, 0].set_ylabel("measured / linear input")
    axes[1, 0].legend([band], ["±1%"], frameon=False, fontsize=9)
    axes[1, 0].set_ylim(*ylim)
    fig.suptitle(f"Power spectrum of the initial conditions:  z = {refhead['redshift']:g}, "
                 f"{box_mpc:g} Mpc/h box, {ngrid_min}³ particles per species, "
                 f"up to the particle Nyquist frequency", fontsize=12, x=0.01, ha="left")
    finish(fig, out)

In [ ]:
plot_matter_pk(ICS, ref=REF)

## 2. Velocities and momenta

There is one column of panels for the dark matter and, if the ICs have gas, one for the gas. The top
panel shows the distribution of the particle speed $|v|$ of each IC and of the reference, as a fraction
of the particles per km/s. The bottom panel shows each distribution divided by the reference's, where
the reference has enough particles per bin.

Velocities are peculiar velocities in km/s. Every particle counts once, whatever its mass. The legend
gives the rms of one component for each IC.

With `momentum=True` the quantity is the momentum of each particle divided by the mean particle mass of
its species, $(m/\langle m\rangle)\,|v|$, also in km/s. Particles have their own mass $m$ if the IC has a
`Mass` block (`PerturbMass = 1`); $\langle m\rangle$ is the header `MassTable`. For ICs with equal particle
masses this is the same as the speed. A `Mass` block is ignored if the header says `PerturbMass = 0`, as
in the power spectrum.

In [ ]:
VEL_TITLES = {DM: "Dark matter", GAS: "Gas"}
# Bins of the reference with fewer particles than this are left out of the ratio panel.
MIN_COUNT = 100


def read_velocities(f, pt):
    """Peculiar velocities of one particle type in km/s."""
    attrs = f["Header"].attrs
    vel = f[f"{pt}/Velocity"][:].astype("f8") * (float(attrs["UnitVelocity_in_cm_per_s"][0]) / 1e5)
    # Without UsePeculiarVelocity the ICs hold v / sqrt(a), the Gadget-2 convention.
    if "UsePeculiarVelocity" in attrs and not int(attrs["UsePeculiarVelocity"][0]):
        vel *= np.sqrt(float(attrs["Time"][0]))
    return vel


def read_relative_mass(f, pt, head):
    """Mass of each particle of one type in units of the mean particle mass (the MassTable).
    None if all particles have the same mass."""
    if f"{pt}/Mass" not in f.blocks or head["perturbmass"] == 0:
        return None
    return f[f"{pt}/Mass"][:].astype("f8") / head["masstable"][pt]


def plot_velocity(ics, ref="ICS_ref_grid/IC", momentum=False, bins=60, ylim=(0.8, 1.2), out=None):
    """Compare the particle speed (or momentum per mean particle mass) distributions of the ICs
    and the reference. out defaults to velocity_comparison.png or momentum_comparison.png."""
    if len(ics) > len(SERIES):
        raise ValueError(f"at most {len(SERIES)} ICs can be compared in one figure")
    out = out or ("momentum_comparison.png" if momentum else "velocity_comparison.png")

    refpath, reffile = open_ic(ref)
    refhead = read_header(reffile)
    opened = [open_ic(p) for p in ics] + [(refpath, reffile)]
    heads = [read_header(f) for _, f in opened[:-1]] + [refhead]
    labels = [label_of(path) for path, _ in opened[:-1]] + [f"{label_of(refpath)} (reference)"]

    # The gas gets its own column only if every IC has gas.
    species = [DM] + ([GAS] if all(GAS in h["types"] for h in heads) else [])

    # values[pt] is one array per IC: the speed, or the momentum per mean particle mass.
    values, sigma = {}, {}
    print(f"box {refhead['box'] * refhead['mpc']:g} Mpc/h, z = {refhead['redshift']:g}: "
          f"{'momentum per mean particle mass' if momentum else 'particle velocities'} in km/s")
    for pt in species:
        values[pt], sigma[pt] = [], []
        print(f"{VEL_TITLES[pt]}:")
        for label, (_, f), head in zip(labels, opened, heads):
            vel = read_velocities(f, pt)
            note = ""
            if momentum:
                relmass = read_relative_mass(f, pt, head)
                if relmass is None:
                    note = "equal masses, so the same as the velocity"
                else:
                    speed = np.linalg.norm(vel, axis=1)
                    note = (f"masses vary by {relmass.std():.2%} rms; velocity rms "
                            f"{np.sqrt(np.mean(vel ** 2)):.3f}; correlation of mass and speed "
                            f"{np.corrcoef(relmass, speed)[0, 1]:+.3f}; net momentum "
                            f"{np.linalg.norm((relmass[:, None] * vel).mean(axis=0)):.2e}")
                    vel = relmass[:, None] * vel
            values[pt].append(np.linalg.norm(vel, axis=1))
            sigma[pt].append(np.sqrt(np.mean(vel ** 2)))
            v = values[pt][-1]
            print(f"  {label}: rms of one component {sigma[pt][-1]:.3f}, mean {v.mean():.3f}, "
                  f"median {np.median(v):.3f}, max {v.max():.3f}" + (f"  ({note})" if note else ""))

    # All distributions share the same bins, so that they can be divided.
    edges = np.linspace(0, max(v.max() for pt in species for v in values[pt]), bins + 1)
    centres = 0.5 * (edges[1:] + edges[:-1])

    ncols = len(species)
    fig, axes = plt.subplots(2, ncols, figsize=(1.5 + 6.5 * ncols, 7.5), sharex=True,
                             sharey="row", squeeze=False, facecolor=SURFACE,
                             gridspec_kw={"height_ratios": [2, 1]}, constrained_layout=True)

    # The reference is drawn last, dashed and in neutral ink, on top of the compared ICs.
    styles = [("-", color, 2) for color in SERIES[:len(labels) - 1]] + [("--", INK, 1.5)]
    for pt, (ax1, ax2) in zip(species, axes.T):
        counts = [np.histogram(v, bins=edges)[0] for v in values[pt]]
        enough = counts[-1] >= MIN_COUNT
        refdensity = counts[-1] / counts[-1].sum() / np.diff(edges)
        for label, count, sig, (ls, color, lw) in zip(labels, counts, sigma[pt], styles):
            density = count / count.sum() / np.diff(edges)
            ax1.plot(centres, density, ls, color=color, lw=lw,
                     label=f"{label}:  σ = {sig:.2f} km/s")
            ax2.plot(centres[enough], density[enough] / refdensity[enough], ls, color=color, lw=lw)
        band = ax2.axhspan(0.95, 1.05, color=GRID, lw=0, zorder=0)
        ax2.axhline(1, color=INK_SECONDARY, lw=1, ls=":")
        ax1.set_title(VEL_TITLES[pt], fontsize=11, loc="left")
        ax1.legend(frameon=False, fontsize=9.5, title="rms of one component", title_fontsize=9.5,
                   alignment="left")
        ax2.set_xlabel(r"momentum / mean particle mass  $(m / \langle m \rangle)\,|v|$  [km/s]"
                       if momentum else r"speed $|v|$  [km/s]")
        for ax in (ax1, ax2):
            style_axes(ax)

    axes[0, 0].set_ylabel("fraction of particles per km/s")
    axes[0, 0].set_ylim(bottom=0)
    axes[1, 0].set_ylabel("distribution / reference")
    axes[1, 0].legend([band], ["±5%"], frameon=False, fontsize=9)
    axes[1, 0].set_ylim(*ylim)
    ngrid = min(h["ngrid"] for h in heads)
    fig.suptitle(f"Particle {'momenta' if momentum else 'velocities'} of the initial conditions:  "
                 f"z = {refhead['redshift']:g}, {refhead['box'] * refhead['mpc']:g} Mpc/h box, "
                 f"{ngrid}³ particles per species", fontsize=12, x=0.01, ha="left")
    finish(fig, out)

In [ ]:
plot_velocity(ICS, ref=REF)

In [ ]:
plot_velocity(ICS, ref=REF, momentum=True)

## 3. Particle masses

Histogram of the particle masses of an IC with perturbed masses (`PerturbMass = 1`). For gas and dark
matter it shows the mass of each particle relative to the mean particle mass of its species (the header
`MassTable`), as a fraction of the particles per percent of mass.

With `PerturbMass = 1` the masses are $m_\mathrm{gas} = \langle m_\mathrm{gas}\rangle (1 + f_c\,\delta_{bc})$
and $m_\mathrm{cdm} = \langle m_\mathrm{cdm}\rangle (1 - f_b\,\delta_{bc})$, so the gas distribution is the
mirror image of the dark matter one, wider by $f_c / f_b$.

In [ ]:
# Dark matter and gas keep the same colours in every panel.
MASS_SPECIES = {DM: ("Dark matter", SERIES[0]), GAS: ("Gas", SERIES[1])}


def plot_mass(ic="ICS_test", bins=200, out="mass_distribution.png"):
    """Histogram of the particle masses of an IC with PerturbMass = 1."""
    path, f = open_ic(ic)
    head = read_header(f)
    attrs = f["Header"].attrs
    if head["perturbmass"] != 1:
        raise ValueError(f"{path}: the header does not say PerturbMass = 1, "
                         f"so all particles of a species have the same mass")
    f_b = float(attrs["OmegaBaryon"][0]) / float(attrs["Omega0"][0])

    # Mass perturbation of each particle in percent: 100 (m / <m> - 1)
    pert = {pt: 100 * (f[f"{pt}/Mass"][:].astype("f8") / head["masstable"][pt] - 1) for pt in head["types"]}
    print(f"{label_of(path)}: particle masses relative to the mean of each species")
    for pt, p in pert.items():
        print(f"  {MASS_SPECIES[pt][0]}: mean mass {head['masstable'][pt]:.6g} (internal units), "
              f"perturbation mean {p.mean():+.1e}%, rms {p.std():.3f}%, "
              f"min {p.min():+.2f}%, max {p.max():+.2f}%")
    if GAS in pert and DM in pert:
        print(f"  rms gas / rms dark matter = {pert[GAS].std() / pert[DM].std():.3f}  "
              f"(expected f_c / f_b = {(1 - f_b) / f_b:.3f})")

    lim = max(np.abs(p).max() for p in pert.values())
    edges = np.linspace(-lim, lim, bins + 1)
    centres = 0.5 * (edges[1:] + edges[:-1])

    fig, ax = plt.subplots(figsize=(8, 5.2), facecolor=SURFACE, constrained_layout=True)
    for pt, p in pert.items():
        name, color = MASS_SPECIES[pt]
        density = np.histogram(p, bins=edges)[0] / len(p) / np.diff(edges)
        ax.fill_between(centres, density, step="mid", color=color, alpha=0.18, lw=0)
        ax.step(centres, density, where="mid", color=color, lw=1.8,
                label=f"{name}:  rms {p.std():.2f}%,  range {p.min():+.1f}% to {p.max():+.1f}%")
    ax.axvline(0, color=INK_SECONDARY, lw=1, ls=":")
    ax.set_xlabel(r"particle mass relative to the mean of its species,  $m / \langle m \rangle - 1$  [%]")
    ax.set_ylabel("fraction of particles per percent")
    ax.set_ylim(0, 1.2 * ax.get_ylim()[1])  # headroom for the legend
    ax.legend(frameon=False, fontsize=10, loc="upper right")
    style_axes(ax)
    fig.suptitle(f"Particle masses of {label_of(path)}:  z = {head['redshift']:g}, "
                 f"{head['box'] * head['mpc']:g} Mpc/h box, {head['ngrid']}³ particles per species",
                 fontsize=12, x=0.01, ha="left")
    finish(fig, out)

In [ ]:
plot_mass(MASS_IC)